# ABC-XYZ и RFM-анализ для итоговой работы

**версия для слушателя**

Этот notebook продолжает проект итоговой аналитической работы после подготовки `bi_dataset.csv`.

Цель: обогатить очищенный датасет сегментационными признаками:

1. ABC-анализ — вклад товаров, категорий или клиентов в выручку.
2. XYZ-анализ — стабильность спроса во времени.
3. ABC-XYZ-матрица — совмещение вклада и стабильности.
4. RFM-анализ — сегментация клиентов по давности, частоте и сумме покупок.
5. Экспорт обогащённого датасета для BI.

Итоговые файлы будут сохранены в папку `outputs`.

## 1. Подготовка окружения

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Рабочая папка:", PROJECT_ROOT)
print("Папка outputs:", OUTPUTS_DIR)
print("pandas:", pd.__version__)

## 2. Загрузка подготовленного датасета

Основной вариант — использовать файл:

```text
outputs/bi_dataset.csv
```

Если он отсутствует, notebook попробует загрузить `data/processed/orders_clean.csv`.

Если и его нет, будет выполнена сборка из исходных CSV-файлов `orders_big.csv`, `clients.csv`, `products.csv`.

In [ ]:
def load_project_dataset():
    bi_path = OUTPUTS_DIR / "bi_dataset.csv"
    clean_path = PROCESSED_DIR / "orders_clean.csv"
    orders_path = RAW_DIR / "orders_big.csv"
    clients_path = RAW_DIR / "clients.csv"
    products_path = RAW_DIR / "products.csv"

    if bi_path.exists():
        print("Загружаем подготовленный BI-датасет:", bi_path)
        df = pd.read_csv(bi_path)
    elif clean_path.exists():
        print("Загружаем очищенный датасет:", clean_path)
        df = pd.read_csv(clean_path)
    elif orders_path.exists():
        print("BI-датасет не найден. Собираем данные из raw-файлов.")
        orders = pd.read_csv(orders_path)
        df = orders.copy()

        if clients_path.exists() and "client_id" in df.columns:
            clients = pd.read_csv(clients_path)
            df = df.merge(clients, on="client_id", how="left")

        if products_path.exists() and "product_id" in df.columns:
            products = pd.read_csv(products_path)
            add_cols = [c for c in ["product_id", "product_name", "brand", "base_price", "cost"] if c in products.columns]
            df = df.merge(products[add_cols], on="product_id", how="left")
    else:
        raise FileNotFoundError(
            "Не найден bi_dataset.csv, orders_clean.csv или orders_big.csv. "
            "Сначала выполните первый notebook проекта."
        )

    if "order_date" in df.columns:
        df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce")

    return df

df = load_project_dataset()

print("Размер датасета:", df.shape)
display(df.head())
display(df.dtypes)

# Нормализация названий полей после merge.
# В разных версиях первого notebook категория может называться category, category_x или category_y.
if "category" not in df.columns:
    if "category_x" in df.columns:
        df["category"] = df["category_x"]
    elif "category_y" in df.columns:
        df["category"] = df["category_y"]

# Аналогично проверяем наличие базовых полей и выводим подсказку.
print("Доступные колонки:", list(df.columns))

## 3. Проверка обязательных полей

Для ABC-XYZ и RFM нужны разные типы полей.

| Метод | Нужные поля |
|---|---|
| ABC | объект анализа и денежный показатель |
| XYZ | объект анализа, дата или период, показатель спроса |
| RFM | клиент, дата покупки, заказ, сумма покупки |

In [ ]:
required_for_project = {
    "ABC": ["category", "revenue"],
    "XYZ": ["category", "order_date", "quantity"],
    "RFM": ["client_id", "order_date", "order_id", "revenue"],
}

for method, cols in required_for_project.items():
    missing = [col for col in cols if col not in df.columns]
    if missing:
        print(f"{method}: не хватает полей {missing}")
    else:
        print(f"{method}: OK")

### Настройте поля под свой датасет

Если вы используете собственный датасет, замените значения переменных ниже.

In [ ]:
# Настройки анализа
abc_entity_col = "category"      # можно заменить на product_id, product_name, client_id
abc_value_col = "revenue"

xyz_entity_col = "category"      # можно заменить на product_id или product_name
xyz_date_col = "order_date"
xyz_value_col = "quantity"

rfm_customer_col = "client_id"
rfm_date_col = "order_date"
rfm_order_col = "order_id"
rfm_value_col = "revenue"

# Часть A. ABC-анализ

ABC-анализ помогает разделить объекты на группы по вкладу в ключевой показатель, чаще всего в выручку.

Учебная логика:

- A — объекты с наибольшим накопленным вкладом, обычно до 80%;
- B — следующий слой, обычно до 95%;
- C — оставшаяся длинная «хвостовая» часть.

Пороги можно менять под предметную область. Главное — объяснить, какие пороги использованы и почему.

In [ ]:
def assign_abc_class(cumulative_share):
    if cumulative_share <= 0.80:
        return "A"
    if cumulative_share <= 0.95:
        return "B"
    return "C"

abc = (
    df
    .dropna(subset=[abc_entity_col, abc_value_col])
    .groupby(abc_entity_col, observed=True)
    .agg(
        total_value=(abc_value_col, "sum"),
        rows_count=(abc_value_col, "count")
    )
    .reset_index()
    .sort_values("total_value", ascending=False)
)

abc["value_share"] = abc["total_value"] / abc["total_value"].sum()
abc["cumulative_share"] = abc["value_share"].cumsum()
abc["abc_class"] = abc["cumulative_share"].apply(assign_abc_class)

display(abc)

abc.to_csv(OUTPUTS_DIR / "abc_analysis.csv", index=False)
print("ABC-анализ сохранён:", OUTPUTS_DIR / "abc_analysis.csv")

In [ ]:
abc_summary = (
    abc
    .groupby("abc_class", as_index=False)
    .agg(
        objects_count=(abc_entity_col, "count"),
        total_value=("total_value", "sum"),
        value_share=("value_share", "sum")
    )
)

display(abc_summary)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(abc[abc_entity_col].astype(str), abc["total_value"])
plt.title("ABC-анализ: вклад объектов в показатель")
plt.xlabel(abc_entity_col)
plt.ylabel(abc_value_col)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Вывод по ABC-анализу

Заполните своими словами:

- Какие объекты попали в класс A:
- Какую долю показателя дают объекты класса A:
- Что означает класс B:
- Что означает класс C:
- Как это можно использовать в BI-дашборде:

# Часть B. XYZ-анализ

XYZ-анализ оценивает стабильность спроса во времени.

В учебном варианте используем коэффициент вариации:

```text
CV = стандартное отклонение спроса / средний спрос
```

Учебные классы:

- X — стабильный спрос;
- Y — умеренно изменчивый спрос;
- Z — нестабильный спрос.

Пороги ниже используются как учебные. В реальном проекте их нужно адаптировать под бизнес-контекст.

In [ ]:
def assign_xyz_class(cv):
    if pd.isna(cv):
        return "no_data"
    if cv <= 0.50:
        return "X"
    if cv <= 1.00:
        return "Y"
    return "Z"

xyz_source = df.dropna(subset=[xyz_entity_col, xyz_date_col, xyz_value_col]).copy()
xyz_source[xyz_date_col] = pd.to_datetime(xyz_source[xyz_date_col], errors="coerce")
xyz_source = xyz_source.dropna(subset=[xyz_date_col])
xyz_source["period"] = xyz_source[xyz_date_col].dt.to_period("M").astype(str)

demand_by_period = (
    xyz_source
    .groupby([xyz_entity_col, "period"], observed=True)
    .agg(demand=(xyz_value_col, "sum"))
    .reset_index()
)

demand_pivot = (
    demand_by_period
    .pivot(index=xyz_entity_col, columns="period", values="demand")
    .fillna(0)
)

xyz = pd.DataFrame({
    xyz_entity_col: demand_pivot.index,
    "mean_demand": demand_pivot.mean(axis=1).values,
    "std_demand": demand_pivot.std(axis=1, ddof=0).values,
})

xyz["cv"] = xyz["std_demand"] / xyz["mean_demand"].replace(0, np.nan)
xyz["xyz_class"] = xyz["cv"].apply(assign_xyz_class)

xyz = xyz.sort_values("cv", ascending=True)

display(xyz)

xyz.to_csv(OUTPUTS_DIR / "xyz_analysis.csv", index=False)
print("XYZ-анализ сохранён:", OUTPUTS_DIR / "xyz_analysis.csv")

In [ ]:
xyz_summary = (
    xyz
    .groupby("xyz_class", as_index=False)
    .agg(
        objects_count=(xyz_entity_col, "count"),
        avg_cv=("cv", "mean"),
        avg_demand=("mean_demand", "mean")
    )
)

display(xyz_summary)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(xyz[xyz_entity_col].astype(str), xyz["cv"])
plt.title("XYZ-анализ: коэффициент вариации спроса")
plt.xlabel(xyz_entity_col)
plt.ylabel("CV")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Вывод по XYZ-анализу

Заполните своими словами:

- Какие объекты имеют наиболее стабильный спрос:
- Какие объекты имеют нестабильный спрос:
- Почему стабильность спроса важна:
- Какие ограничения есть у расчёта:

# Часть C. ABC-XYZ-матрица

ABC-XYZ-матрица объединяет два взгляда:

- ABC показывает вклад в выручку или другой денежный показатель;
- XYZ показывает стабильность спроса.

Например:

- AX — важные и стабильные объекты;
- AZ — важные, но нестабильные объекты;
- CZ — низкий вклад и нестабильный спрос.

In [ ]:
abc_xyz = abc.merge(
    xyz[[xyz_entity_col, "mean_demand", "std_demand", "cv", "xyz_class"]],
    left_on=abc_entity_col,
    right_on=xyz_entity_col,
    how="left"
)

if xyz_entity_col != abc_entity_col:
    abc_xyz = abc_xyz.drop(columns=[xyz_entity_col])

abc_xyz["abc_xyz_class"] = abc_xyz["abc_class"].astype(str) + abc_xyz["xyz_class"].astype(str)

display(abc_xyz)

abc_xyz.to_csv(OUTPUTS_DIR / "abc_xyz_matrix.csv", index=False)
print("ABC-XYZ-матрица сохранена:", OUTPUTS_DIR / "abc_xyz_matrix.csv")

In [ ]:
abc_xyz_summary = (
    abc_xyz
    .groupby(["abc_class", "xyz_class", "abc_xyz_class"], as_index=False)
    .agg(
        objects_count=(abc_entity_col, "count"),
        total_value=("total_value", "sum"),
        avg_cv=("cv", "mean")
    )
    .sort_values(["abc_class", "xyz_class"])
)

display(abc_xyz_summary)

### Вывод по ABC-XYZ-матрице

Заполните своими словами:

- Какие объекты попали в AX:
- Какие объекты попали в AZ:
- Какие объекты требуют управленческого внимания:
- Какие классы стоит вынести на BI-дашборд:

# Часть D. RFM-анализ клиентов

RFM-анализ сегментирует клиентов по трём признакам:

- Recency — как давно клиент покупал;
- Frequency — как часто клиент покупает;
- Monetary — сколько денег принёс клиент.

В учебном варианте используем шкалу 1–5 для каждого показателя.

In [ ]:
rfm_source = df.dropna(subset=[rfm_customer_col, rfm_date_col, rfm_order_col, rfm_value_col]).copy()
rfm_source[rfm_date_col] = pd.to_datetime(rfm_source[rfm_date_col], errors="coerce")
rfm_source = rfm_source.dropna(subset=[rfm_date_col])

snapshot_date = rfm_source[rfm_date_col].max() + pd.Timedelta(days=1)

rfm = (
    rfm_source
    .groupby(rfm_customer_col)
    .agg(
        recency=(rfm_date_col, lambda x: (snapshot_date - x.max()).days),
        frequency=(rfm_order_col, "nunique"),
        monetary=(rfm_value_col, "sum")
    )
    .reset_index()
)

display(rfm.head())
print("Количество клиентов в RFM:", len(rfm))

In [ ]:
def qcut_score(series, q=5, reverse=False):
    ranked = series.rank(method="first")
    labels = list(range(1, q + 1))
    if reverse:
        labels = labels[::-1]
    return pd.qcut(ranked, q=q, labels=labels).astype(int)

# Для recency меньше = лучше, поэтому reverse=True.
rfm["r_score"] = qcut_score(rfm["recency"], q=5, reverse=True)
rfm["f_score"] = qcut_score(rfm["frequency"], q=5, reverse=False)
rfm["m_score"] = qcut_score(rfm["monetary"], q=5, reverse=False)

rfm["rfm_score"] = (
    rfm["r_score"].astype(str) +
    rfm["f_score"].astype(str) +
    rfm["m_score"].astype(str)
)

rfm["rfm_total"] = rfm["r_score"] + rfm["f_score"] + rfm["m_score"]

display(rfm.head())

In [ ]:
def assign_rfm_segment(row):
    r, f, m = row["r_score"], row["f_score"], row["m_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "champions"
    if r >= 4 and f >= 3:
        return "loyal_recent"
    if r >= 3 and m >= 4:
        return "big_spenders"
    if r <= 2 and f >= 4:
        return "at_risk_loyal"
    if r <= 2 and m >= 4:
        return "at_risk_high_value"
    if r <= 2 and f <= 2 and m <= 2:
        return "hibernating"
    if r >= 4 and f <= 2:
        return "new_or_promising"
    return "regular"

rfm["rfm_segment"] = rfm.apply(assign_rfm_segment, axis=1)

display(rfm.head())

rfm.to_csv(OUTPUTS_DIR / "rfm_analysis.csv", index=False)
print("RFM-анализ сохранён:", OUTPUTS_DIR / "rfm_analysis.csv")

In [ ]:
rfm_segment_summary = (
    rfm
    .groupby("rfm_segment", as_index=False)
    .agg(
        clients_count=(rfm_customer_col, "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        total_monetary=("monetary", "sum"),
        avg_monetary=("monetary", "mean")
    )
    .sort_values("total_monetary", ascending=False)
)

display(rfm_segment_summary)
rfm_segment_summary.to_csv(OUTPUTS_DIR / "rfm_segment_summary.csv", index=False)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(rfm_segment_summary["rfm_segment"], rfm_segment_summary["clients_count"])
plt.title("Количество клиентов по RFM-сегментам")
plt.xlabel("RFM-сегмент")
plt.ylabel("Количество клиентов")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Вывод по RFM-анализу

Заполните своими словами:

- Какие сегменты клиентов самые многочисленные:
- Какие сегменты дают больше денежного вклада:
- Какие клиенты требуют реактивации:
- Какие клиенты выглядят наиболее ценными:
- Как RFM-сегменты можно использовать в BI:

# Часть E. Экспорт обогащённого датасета для BI

Добавим к исходным строкам:

- ABC-класс;
- XYZ-класс;
- ABC-XYZ-класс;
- RFM-сегмент клиента.

Этот файл можно использовать в BI-дашборде для фильтров и сегментации.

In [ ]:
segmented_bi_dataset = df.copy()

# Добавляем ABC-XYZ по выбранному объекту.
abc_xyz_features = abc_xyz[
    [abc_entity_col, "abc_class", "xyz_class", "abc_xyz_class"]
].copy()

segmented_bi_dataset = segmented_bi_dataset.merge(
    abc_xyz_features,
    on=abc_entity_col,
    how="left"
)

# Добавляем RFM по клиенту.
rfm_features = rfm[
    [rfm_customer_col, "recency", "frequency", "monetary", "r_score", "f_score", "m_score", "rfm_score", "rfm_total", "rfm_segment"]
].copy()

segmented_bi_dataset = segmented_bi_dataset.merge(
    rfm_features,
    on=rfm_customer_col,
    how="left"
)

segmented_path = OUTPUTS_DIR / "segmented_bi_dataset.csv"
segmented_bi_dataset.to_csv(segmented_path, index=False)

print("Обогащённый BI-датасет сохранён:", segmented_path)
print("Размер:", segmented_bi_dataset.shape)
display(segmented_bi_dataset.head())

# Часть F. Предварительные выводы

In [ ]:
conclusions = f'''
# Выводы по ABC-XYZ и RFM-анализу

## 1. ABC-анализ
Класс A включает объекты с наибольшим вкладом в показатель `{abc_value_col}`. Эти объекты стоит показывать на BI-дашборде отдельно и использовать как приоритетные для анализа.

## 2. XYZ-анализ
XYZ-анализ показывает стабильность спроса. Объекты класса X имеют более стабильный спрос, объекты класса Z требуют осторожной интерпретации и дополнительной проверки.

## 3. ABC-XYZ-матрица
ABC-XYZ-матрица помогает разделить объекты не только по денежному вкладу, но и по стабильности. Наиболее управленчески значимы классы AX, AY и AZ.

## 4. RFM-анализ
RFM-анализ разделяет клиентов по давности, частоте и сумме покупок. Наиболее ценные сегменты можно использовать для отдельного анализа в BI.

## 5. Ограничения
Результаты зависят от выбранных порогов ABC, XYZ и RFM. Пороговые значения нужно объяснить в итоговой работе.

## 6. Следующий шаг
Загрузить `segmented_bi_dataset.csv` в BI-систему и построить дашборд с фильтрами по ABC-классу, XYZ-классу и RFM-сегменту.
'''

conclusions_path = OUTPUTS_DIR / "abc_xyz_rfm_conclusions.md"

with open(conclusions_path, "w", encoding="utf-8") as file:
    file.write(conclusions)

print("Файл с выводами создан:", conclusions_path)
print(conclusions)

## Финальный чек-лист

Проверьте, что созданы файлы:

- [ ] `outputs/abc_analysis.csv`
- [ ] `outputs/xyz_analysis.csv`
- [ ] `outputs/abc_xyz_matrix.csv`
- [ ] `outputs/rfm_analysis.csv`
- [ ] `outputs/rfm_segment_summary.csv`
- [ ] `outputs/segmented_bi_dataset.csv`
- [ ] `outputs/abc_xyz_rfm_conclusions.md`

В итоговой работе нужно кратко объяснить:

- какие поля использовались для ABC;
- какие поля использовались для XYZ;
- какие поля использовались для RFM;
- какие пороги применялись;
- какие выводы получены;
- какие ограничения есть у сегментации.